In [ ]:
import os
import json
import time
from datetime import datetime, timedelta
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from dotenv import load_dotenv

load_dotenv()

# --- Configuration ---
API_KEY = os.getenv("API_KEY")
query = 'war ukraine'  #'zelensky trump'
max_results_per_search = 50
max_videos = 2000
output_json = 'youtube_war_ukraine_5.json'

# --- TEMPORAL FILTERS ---
published_after = '2025-05-01T00:00:00Z'
published_before = '2025-05-10T23:59:59Z'

# --- SETTINGS ---
request_delay = 1.0
comment_score_min = 5
max_comments_per_video = 200

def initialize_youtube_api():
    """Initialize the YouTube API client"""
    if not API_KEY:
        print("❌ Error: YOUTUBE_API_KEY not found in .env")
        exit(1)
    
    try:
        youtube = build('youtube', 'v3', developerKey=API_KEY)
        print("✅ YouTube API initialized successfully!")
        return youtube
    except Exception as e:
        print(f"❌ Error initializing YouTube API: {e}")
        exit(1)

def search_videos(youtube, query, published_after=None, published_before=None, max_results=50, page_token=None):
    """Search for videos with time filters"""
    try:
        search_params = {
            'q': query,
            'part': 'id,snippet',
            'type': 'video',
            'maxResults': max_results,
            'order': 'date',
            'regionCode': 'US'
        }
        
        # Add time filters if specified
        if published_after:
            search_params['publishedAfter'] = published_after
        if published_before:
            search_params['publishedBefore'] = published_before
        if page_token:
            search_params['pageToken'] = page_token
            
        response = youtube.search().list(**search_params).execute()
        return response
    
    except HttpError as e:
        print(f"❌ HTTP error during search: {e}")
        return None
    except Exception as e:
        print(f"❌ Generic error during search: {e}")
        return None

def get_video_details(youtube, video_ids):
    """Get video details (statistics, duration, etc.)"""
    try:
        response = youtube.videos().list(
            part='statistics,contentDetails,snippet',
            id=','.join(video_ids)
        ).execute()
        return response
    except HttpError as e:
        print(f"❌ Error getting video details: {e}")
        return None

def get_video_comments(youtube, video_id, max_results=100):
    """Get video comments with pagination"""
    comments = []
    next_page_token = None
    
    try:
        while len(comments) < max_results:
            remaining = max_results - len(comments)
            per_page = min(100, remaining)
            
            request_params = {
                'part': 'snippet,replies',
                'videoId': video_id,
                'maxResults': per_page,
                'order': 'relevance', 
                'textFormat': 'plainText'
            }
            
            if next_page_token:
                request_params['pageToken'] = next_page_token
            
            response = youtube.commentThreads().list(**request_params).execute()
            
            for item in response['items']:
                comment = item['snippet']['topLevelComment']['snippet']
                
                # Filter by minimum score
                if comment.get('likeCount', 0) >= comment_score_min:
                    comment_data = {
                        'author': comment.get('authorDisplayName', '[Unknown]'),
                        'text': comment.get('textDisplay', ''),
                        'like_count': comment.get('likeCount', 0),
                        'published_at': comment.get('publishedAt', ''),
                        'updated_at': comment.get('updatedAt', '')
                    }
                    
                    # Add replies if present
                    if 'replies' in item:
                        replies = []
                        for reply in item['replies']['comments']:
                            reply_snippet = reply['snippet']
                            if reply_snippet.get('likeCount', 0) >= comment_score_min:
                                replies.append({
                                    'author': reply_snippet.get('authorDisplayName', '[Unknown]'),
                                    'text': reply_snippet.get('textDisplay', ''),
                                    'like_count': reply_snippet.get('likeCount', 0),
                                    'published_at': reply_snippet.get('publishedAt', '')
                                })
                        comment_data['replies'] = replies
                    
                    comments.append(comment_data)
            
            # Check if there are more pages
            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
                
            # Pause between requests
            time.sleep(0.5)
    
    except HttpError as e:
        if 'commentsDisabled' in str(e):
            print(f"      ℹ️ Comments disabled for video {video_id}")
        else:
            print(f"      ⚠️ Error getting comments for {video_id}: {e}")
    except Exception as e:
        print(f"      ⚠️ Generic error getting comments for {video_id}: {e}")
    
    return comments

def main():
    print("🎥 YouTube Data Scraper - Starting data collection...")
    print(f"🔍 Query: '{query}'")
    print(f"📅 Time range: {published_after} -> {published_before}")
    
    # Initialize API
    youtube = initialize_youtube_api()
    
    all_data = []
    total_videos = 0
    total_comments = 0
    next_page_token = None
    page_number = 1
    
    # Quota usage estimate
    estimated_quota = 0
    
    while total_videos < max_videos:
        print(f"\n📄 Page {page_number} - Searching videos...")
        
        # Compute how many videos are missing
        remaining_videos = max_videos - total_videos
        search_limit = min(max_results_per_search, remaining_videos)
        
        # Search videos
        search_response = search_videos(
            youtube, query, published_after, published_before, 
            search_limit, next_page_token
        )
        
        if not search_response or not search_response.get('items'):
            print("   ℹ️ No videos found or end of results")
            break
        
        estimated_quota += 100
        videos = search_response['items']
        print(f"   📹 Found {len(videos)} videos")
        
        # Get video details (statistics)
        video_ids = [video['id']['videoId'] for video in videos]
        video_details_response = get_video_details(youtube, video_ids)
        estimated_quota += 1

        # Create map for quick access to details
        video_details_map = {}
        if video_details_response:
            for detail in video_details_response['items']:
                video_details_map[detail['id']] = detail
        
        page_videos = 0
        page_comments = 0
        
        # Process each video
        for video in videos:
            video_id = video['id']['videoId']
            snippet = video['snippet']
            
            # Get additional details
            details = video_details_map.get(video_id, {})
            stats = details.get('statistics', {})
            content_details = details.get('contentDetails', {})
            
            print(f"   🎬 Processing: {snippet['title'][:50]}...")
            
            # Get comments
            comments = get_video_comments(youtube, video_id, max_comments_per_video)
            estimated_quota += max(1, len(comments) // 100)
            
            # Build video record
            video_record = {
                'page_number': page_number,
                'video_id': video_id,
                'title': snippet['title'],
                'description': snippet.get('description', '')[:500],
                'channel_title': snippet['channelTitle'],
                'channel_id': snippet['channelId'],
                'published_at': snippet['publishedAt'],
                'thumbnail_url': snippet['thumbnails']['high']['url'] if 'high' in snippet['thumbnails'] else '',
                
                # Statistics
                'view_count': int(stats.get('viewCount', 0)),
                'like_count': int(stats.get('likeCount', 0)),
                'comment_count': int(stats.get('commentCount', 0)),
                'duration': content_details.get('duration', ''),
                
                # Collected comments
                'comments': comments,
                'comments_collected': len(comments)
            }
            
            all_data.append(video_record)
            page_videos += 1
            page_comments += len(comments)
            
            # Small pause between videos
            time.sleep(0.2)
        
        total_videos += page_videos
        total_comments += page_comments
        
        print(f"   ✅ {page_videos} videos, {page_comments} comments collected")
        print(f"   📊 Totals: {total_videos} videos, {total_comments} comments")
        print(f"   📈 Estimated quota used: {estimated_quota}/10000")
        
        # Check for more pages
        next_page_token = search_response.get('nextPageToken')
        if not next_page_token:
            print("   ℹ️ No more pages available")
            break
        
        page_number += 1
        
        # Pause between pages to respect rate limits
        time.sleep(request_delay)
        
        # Save progress every 5 pages
        if page_number % 5 == 0:
            print(f"   💾 Saving progress...")
            with open(f"temp_{output_json}", 'w', encoding='utf-8') as f:
                json.dump(all_data, f, ensure_ascii=False, indent=2)
    
    # Final save
    print(f"\n💾 Final save to {output_json}...")
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)
    
    # Final stats
    print(f"\n🎉 COMPLETED!")
    print(f"📊 Final statistics:")
    print(f"   📄 Pages processed: {page_number}")
    print(f"   🎬 Total videos: {total_videos}")
    print(f"   💬 Total comments: {total_comments}")
    print(f"   📈 Estimated quota used: {estimated_quota}/10000 ({(estimated_quota/10000)*100:.1f}%)")
    print(f"   📄 File saved: {output_json}")
    
    if total_videos > 0:
        print(f"   ⏱️ Avg comments per video: {total_comments/total_videos:.1f}")
    
    # Remove temp file
    temp_file = f"temp_{output_json}"
    if os.path.exists(temp_file):
        os.remove(temp_file)
        print(f"🗑️ Temporary file removed")
    
    print(f"\n✨ Collection completed!")

if __name__ == "__main__":
    main()


In [ ]:
import json
import pandas as pd
from datetime import datetime

#  --- Load JSON file ---
with open('youtube_war_ukraine_5.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# --- Statistics ---
num_videos = len(data)
num_comments = sum(video['comments_collected'] for video in data)
total_views = sum(video['view_count'] for video in data)
total_likes = sum(video['like_count'] for video in data)

print(f"🎥 Total number of videos: {num_videos}")
print(f"💬 Total number of comments: {num_comments}")
print(f"👀 Total number of views: {total_views:,}")
print(f"👍 Total number of likes: {total_likes:,}")

if num_videos > 0:
    print(f"📊 Average comments per video: {num_comments/num_videos:.1f}")
    print(f"📊 Average views per video: {total_views/num_videos:,.0f}")

# --- DataFrame creation ---
rows = []

for video in data:
    video_id = video['video_id']
    video_title = video['title']
    channel_title = video['channel_title']
    channel_id = video['channel_id']
    published_at = video['published_at']
    view_count = video['view_count']
    like_count = video['like_count']
    comment_count = video['comment_count']
    duration = video['duration']
    description = video['description']

    if not video['comments']:
        rows.append({
            'video_id': video_id,
            'video_title': video_title,
            'channel_title': channel_title,
            'channel_id': channel_id,
            'video_published_at': published_at,
            'video_view_count': view_count,
            'video_like_count': like_count,
            'video_comment_count': comment_count,
            'video_duration': duration,
            'video_description': description[:200] + '...' if len(description) > 200 else description,
            
            'comment_author': None,
            'comment_text': None,
            'comment_like_count': None,
            'comment_published_at': None,
            'comment_updated_at': None,
            'comment_replies_count': None,
            'is_reply': False
        })
    else:
        # Process every comment
        for comment in video['comments']:
            rows.append({
                'video_id': video_id,
                'video_title': video_title,
                'channel_title': channel_title,
                'channel_id': channel_id,
                'video_published_at': published_at,
                'video_view_count': view_count,
                'video_like_count': like_count,
                'video_comment_count': comment_count,
                'video_duration': duration,
                'video_description': description[:200] + '...' if len(description) > 200 else description,
                
                'comment_author': comment['author'],
                'comment_text': comment['text'],
                'comment_like_count': comment['like_count'],
                'comment_published_at': comment['published_at'],
                'comment_updated_at': comment['updated_at'],
                'comment_replies_count': len(comment.get('replies', [])),
                'is_reply': False
            })
            
            # Process answers if available
            if 'replies' in comment and comment['replies']:
                for reply in comment['replies']:
                    rows.append({
                        'video_id': video_id,
                        'video_title': video_title,
                        'channel_title': channel_title,
                        'channel_id': channel_id,
                        'video_published_at': published_at,
                        'video_view_count': view_count,
                        'video_like_count': like_count,
                        'video_comment_count': comment_count,
                        'video_duration': duration,
                        'video_description': description[:200] + '...' if len(description) > 200 else description,
                        
                        # Dati reply
                        'comment_author': reply['author'],
                        'comment_text': reply['text'],
                        'comment_like_count': reply['like_count'],
                        'comment_published_at': reply['published_at'],
                        'comment_updated_at': reply.get('updated_at', reply['published_at']),
                        'comment_replies_count': 0,
                        'is_reply': True
                    })

# Create DataFrame
df = pd.DataFrame(rows)

# --- More processing session ---
if not df.empty:
    # Convert datatime
    df['video_published_at'] = pd.to_datetime(df['video_published_at'])
    df['comment_published_at'] = pd.to_datetime(df['comment_published_at'])

    df['video_published_at'] = df['video_published_at'].dt.tz_localize(None)

    df['video_age_days'] = (datetime.now() - df['video_published_at']).dt.days
    df['comment_length'] = df['comment_text'].str.len()

    def parse_duration(duration_str):
        if not duration_str or duration_str == '':
            return 0
        try:
            duration_str = duration_str.replace('PT', '')
            minutes = 0
            seconds = 0
            
            if 'H' in duration_str:
                parts = duration_str.split('H')
                hours = int(parts[0])
                duration_str = parts[1]
                minutes += hours * 60
            
            if 'M' in duration_str:
                parts = duration_str.split('M')
                minutes += int(parts[0])
                duration_str = parts[1]
            
            if 'S' in duration_str:
                seconds = int(duration_str.replace('S', ''))
            
            return minutes * 60 + seconds
        except:
            return 0
    
    df['video_duration_seconds'] = df['video_duration'].apply(parse_duration)
    df['video_duration_minutes'] = df['video_duration_seconds'] / 60

# --- DataFrame Visualization ---
print(f"\n🧾 DataFrame created with {len(df)} rows")
print(f"📊 Available columns: {list(df.columns)}")

if not df.empty:
    print("\n📋 DataFrame sample:")
    print(df[['video_title', 'channel_title', 'comment_author', 'comment_like_count', 'is_reply']].head())
    
    print(f"\n📈 Quick statistics:")
    print(f"   🎬 Unique videos: {df['video_id'].nunique()}")
    print(f"   📺 Unique channels: {df['channel_id'].nunique()}")
    print(f"   👤 Unique comment authors: {df['comment_author'].nunique()}")
    print(f"   💬 Top-level comments: {len(df[df['is_reply'] == False])}")
    print(f"   🔄 Replies: {len(df[df['is_reply'] == True])}")
    
    if df['comment_like_count'].notna().any():
        print(f"   👍 Average likes per comment: {df['comment_like_count'].mean():.1f}")
        print(f"   👍 Most liked comment: {df['comment_like_count'].max()}")
    
    # Top channels by number of videos
    print(f"\n🏆 Top 5 channels by number of videos:")
    top_channels = df.groupby('channel_title')['video_id'].nunique().sort_values(ascending=False).head()
    for channel, count in top_channels.items():
        print(f"   📺 {channel}: {count} videos")

# --- Saving ---
if not df.empty:
    # Save as CSV
    csv_filename = 'youtube_war_ukraine_data_5.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"\n💾 DataFrame saved as: {csv_filename}")
    
    # Save a dataset with only comments (excluding videos without comments)
    df_comments_only = df[df['comment_text'].notna()]
    if not df_comments_only.empty:
        comments_csv = 'youtube_war_ukraine_comments_only_5.csv'
        df_comments_only.to_csv(comments_csv, index=False, encoding='utf-8')
        print(f"💾 Comments only saved as: {comments_csv}")
        
    # Create a summary of videos
    video_summary = df.groupby(['video_id', 'video_title', 'channel_title']).agg({
        'video_view_count': 'first',
        'video_like_count': 'first',
        'video_comment_count': 'first',
        'video_duration_minutes': 'first',
        'comment_like_count': ['count', 'sum', 'mean'],
        'video_published_at': 'first'
    }).reset_index()
    
    # Flatten column names
    video_summary.columns = ['video_id', 'video_title', 'channel_title', 'view_count', 
                           'like_count', 'total_comments', 'duration_minutes',
                           'comments_collected', 'total_comment_likes', 'avg_comment_likes', 'published_at']
    
    video_summary_csv = 'youtube_war_ukraine_video_summary_5.csv'
    video_summary.to_csv(video_summary_csv, index=False, encoding='utf-8')
    print(f"💾 Video summary saved as: {video_summary_csv}")

print(f"\n✨ Processing completed!")